Cell 1: Setup & Imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers torch safetensors pytorch-crf seqeval scikit-learn

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from safetensors.torch import load_file
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from seqeval.metrics import precision_score, recall_score, f1_score as seq_f1_score, classification_report
import json
import random
import copy

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Cell 2: Constants and Device Setup

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
drive_path = '/content/drive/MyDrive/intent_project'

# Held-out zero-shot languages requiring evaluation
zero_shot_languages = ['sw', 'ur', 'te', 'ta', 'cy']  # Swahili, Urdu, Telugu, Tamil, Welsh

# Few-shot sample size per language for fine-tuning
few_shot_sample_size = 50


Cell 3: Load Final Models

In [3]:
# Load intent classification model
intent_model_dir = f'{drive_path}/xlm-roberta-intent-classifier-final'
intent_tokenizer = AutoTokenizer.from_pretrained(intent_model_dir)
intent_model = AutoModelForSequenceClassification.from_pretrained(intent_model_dir)
intent_model.to(device)
intent_model.eval()

with open(f'{intent_model_dir}/intent2id.json', 'r') as f:
    intent2id = json.load(f)
with open(f'{intent_model_dir}/id2intent.json', 'r') as f:
    id2intent = json.load(f)

# Load slot filling CRF model
class XLMRobertaWithCRF(torch.nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id):
        super().__init__()
        from transformers import AutoModel
        from torchcrf import CRF
        self.transformer = AutoModel.from_pretrained(model_name)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(self.transformer.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.id2label = id2label
        self.label2id = label2id
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        sequence_output = outputs[0]
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)
        if labels is not None:
            labels = torch.where(labels == -100, torch.zeros_like(labels), labels)
            loss = -self.crf(logits, labels, mask=attention_mask.bool(), reduction='mean')
            return {'loss': loss, 'logits': logits}
        else:
            predictions = self.crf.decode(logits, mask=attention_mask.bool())
            return {'logits': logits, 'predictions': predictions}

slot_model_dir = f'{drive_path}/slot_filling_model_crf/final_model'
with open(f"{slot_model_dir}/id2label.json", 'r') as f:
    id2slot = json.load(f)
with open(f"{slot_model_dir}/label2id.json", 'r') as f:
    slot2id = json.load(f)

slot_tokenizer = AutoTokenizer.from_pretrained(slot_model_dir)
slot_model = XLMRobertaWithCRF(model_name="xlm-roberta-base",
                              num_labels=len(slot2id),
                              id2label=id2slot,
                              label2id=slot2id)
model_state = load_file(f"{slot_model_dir}/model.safetensors")
slot_model.load_state_dict(model_state)
slot_model.to(device)
slot_model.eval()

print(f"Loaded intent and slot filling models.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loaded intent and slot filling models.


Cell 4: Utility Functions for Inference and BIO Conversion

In [16]:
def predict_intents_batch(texts, batch_size=16):
    all_preds = []
    all_confs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = intent_tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True).to(device)
        with torch.no_grad():
            logits = intent_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = logits.argmax(dim=-1).cpu().numpy()
        for pred_id, prob_vec in zip(preds, probs):
            all_preds.append(id2intent[str(pred_id)])
            all_confs.append(prob_vec[pred_id])
    return all_preds, all_confs

def predict_slots_batch(texts, batch_size=8):
    all_spans = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_words = [text.split() for text in batch_texts]
        inputs = slot_tokenizer(batch_words, is_split_into_words=True, return_tensors='pt', truncation=True, padding='max_length').to(device)
        with torch.no_grad():
            outputs = slot_model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        batch_preds = outputs['predictions']
        for words, preds in zip(batch_words, batch_preds):
            spans = []
            current_slot = None
            current_words = []
            for w, p in zip(words, preds):
                label = id2slot[str(p)]
                if label == 'O':
                    if current_slot is not None:
                        spans.append({"type": current_slot, "value": " ".join(current_words)})
                        current_slot = None
                        current_words = []
                elif label.startswith('B-'):
                    if current_slot is not None:
                        spans.append({"type": current_slot, "value": " ".join(current_words)})
                    current_slot = label[2:]
                    current_words = [w]
                elif label.startswith('I-') and current_slot == label[2:]:
                    current_words.append(w)
                else:
                    if current_slot is not None:
                        spans.append({"type": current_slot, "value": " ".join(current_words)})
                    current_slot = label[2:]
                    current_words = [w]
            if current_slot is not None:
                spans.append({"type": current_slot, "value": " ".join(current_words)})
            all_spans.append(spans)
    return all_spans


In [18]:
import json

def save_results(output_path, results):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

def load_results(input_path):
    with open(input_path, "r", encoding="utf-8") as f:
        return json.load(f)

Cell 5: Load and Prepare Test Data (Zero-shot and Few-shot Splits)

In [7]:
import os
import json
import random

data_folder = '/content/drive/MyDrive/intent_project/data'  # Your actual folder path

all_data = []
data_by_lang = {}

for filename in os.listdir(data_folder):
    if filename.endswith('.jsonl'):
        lang_code = filename.split('.')[0].split('-')[0]  # e.g., 'af' from 'af-ZA.jsonl'
        with open(os.path.join(data_folder, filename), 'r', encoding='utf-8') as f:
            for line in f:
                example = json.loads(line.strip())
                example['language'] = lang_code
                all_data.append(example)
                if lang_code not in data_by_lang:
                    data_by_lang[lang_code] = []
                data_by_lang[lang_code].append(example)

print(f"Loaded {len(all_data)} examples from {len(data_by_lang)} languages.")

zero_shot_languages = ['sw', 'ur', 'te', 'ta', 'cy']

# Prepare zero-shot test set
test_data_zeroshot = []
for lang in zero_shot_languages:
    test_data_zeroshot.extend(data_by_lang.get(lang, []))

# Prepare train/val/test for non-zero-shot languages
train_data, val_data, test_data = [], [], []
for lang, examples in data_by_lang.items():
    if lang in zero_shot_languages:
        continue
    random.shuffle(examples)
    n = len(examples)
    train_end = int(0.8 * n)
    val_end = int(0.9 * n)
    train_data.extend(examples[:train_end])
    val_data.extend(examples[train_end:val_end])
    test_data.extend(examples[val_end:])

# Prepare few-shot support and eval sets
few_shot_support = []
few_shot_eval = []
for lang in zero_shot_languages:
    examples = data_by_lang.get(lang, [])
    random.shuffle(examples)
    if len(examples) > 50:
        few_shot_support.extend(examples[:50])
        few_shot_eval.extend(examples[50:])
    else:
        few_shot_support.extend(examples)  # if less than 50
        few_shot_eval.extend([])

print("Data Split Summary:")
print(f"Train examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
print(f"Test examples (non-zero-shot): {len(test_data)}")
print(f"Zero-shot test examples: {len(test_data_zeroshot)}")
print(f"Few-shot support: {len(few_shot_support)}")
print(f"Few-shot evaluation: {len(few_shot_eval)}")


Loaded 842571 examples from 50 languages.
Data Split Summary:
Train examples: 607937
Validation examples: 75992
Test examples (non-zero-shot): 76037
Zero-shot test examples: 82605
Few-shot support: 250
Few-shot evaluation: 82355


In [8]:
import json

splits_to_save = {
    "train_data.json": train_data,
    "val_data.json": val_data,
    "test_data.json": test_data,
    "zero_shot_test_data.json": test_data_zeroshot,
    "few_shot_support.json": few_shot_support,
    "few_shot_eval.json": few_shot_eval
}

out_folder = "/content/drive/MyDrive/intent_project/splits/"
os.makedirs(out_folder, exist_ok=True)

for fname, data in splits_to_save.items():
    with open(os.path.join(out_folder, fname), 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Saved {fname} ({len(data)} examples)")

Saved train_data.json (607937 examples)
Saved val_data.json (75992 examples)
Saved test_data.json (76037 examples)
Saved zero_shot_test_data.json (82605 examples)
Saved few_shot_support.json (250 examples)
Saved few_shot_eval.json (82355 examples)


Cell 6: Few-Shot Fine-tuning Function

In [10]:
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
import torch.nn as nn

class IntentSlotDataset(Dataset):
    def __init__(self, data, intent_tokenizer, slot_tokenizer, label_maps):
        self.data = data
        self.intent_tokenizer = intent_tokenizer
        self.slot_tokenizer = slot_tokenizer
        self.id2slot = label_maps['id2slot']
        self.slot2id = label_maps['slot2id']
        self.intent2id = label_maps['intent2id']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        intent_label = self.intent2id[example['intent']]
        slots = example['slots']
        utterance = example['utterance']

        intent_inputs = self.intent_tokenizer(utterance, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
        slot_words = utterance.split()
        slot_inputs = self.slot_tokenizer(slot_words, is_split_into_words=True, truncation=True, padding='max_length', max_length=128, return_tensors='pt')

        # Convert slots into label ids aligned to words for slot filling
        slot_label_ids = [self.slot2id['O']] * slot_inputs['input_ids'].shape[1]  # max length
        bio_tags = slots_to_bio_tags(slot_words, slots)
        for idx_w, tag in enumerate(bio_tags):
            slot_label_ids[idx_w] = self.slot2id.get(tag, self.slot2id['O'])

        slot_label_ids = torch.tensor(slot_label_ids)

        return {
            'intent_inputs': intent_inputs,
            'slot_inputs': slot_inputs,
            'intent_label': torch.tensor(intent_label),
            'slot_label': slot_label_ids
        }

def fine_tune_few_shot(model_intent, model_slot, dataset, epochs=3, batch_size=8, lr=2e-5):
    model_intent.train()
    model_slot.train()

    optimizer = AdamW(list(model_intent.parameters()) + list(model_slot.parameters()), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        for batch in dataloader:
            optimizer.zero_grad()

            intent_inputs = batch['intent_inputs']
            slot_inputs = batch['slot_inputs']
            intent_label = batch['intent_label'].to(device)
            slot_label = batch['slot_label'].to(device)

            # Intent forward
            intent_outputs = model_intent(**{k: v.squeeze(1).to(device) for k, v in intent_inputs.items()})
            intent_loss = loss_fn(intent_outputs.logits, intent_label)

            # Slot forward
            slot_outputs = model_slot(input_ids=slot_inputs['input_ids'].squeeze(1).to(device),
                                     attention_mask=slot_inputs['attention_mask'].squeeze(1).to(device),
                                     labels=slot_label)
            slot_loss = slot_outputs['loss']

            total_loss = intent_loss + slot_loss
            total_loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}/{epochs} done, loss: {total_loss.item():.4f}")

    model_intent.eval()
    model_slot.eval()


Cell 7: Evaluation Helper

In [17]:
import re

def evaluate_with_preds(data, pred_intents, pred_slots):
    y_true_intent = []
    y_true_slots = []

    for example in data:
        true_intent = example['intent']
        slot_matches = re.findall(r"\[(\w+)\s*:\s*([^\]]+)\]", example.get("annot_utt", ""))
        true_slots = [{"type": slot_type.strip(), "value": slot_val.strip()} for slot_type, slot_val in slot_matches]
        y_true_intent.append(true_intent)
        words = example['utt'].split()
        true_slot_labels = slots_to_bio_tags(words, true_slots)
        y_true_slots.append(true_slot_labels)

    y_pred_intent = pred_intents
    y_pred_slots = []
    for example, slots in zip(data, pred_slots):
        words = example['utt'].split()
        bio_tags = slots_to_bio_tags(words, slots)
        y_pred_slots.append(bio_tags)

    print("Intent metrics:")
    print(f"Accuracy: {accuracy_score(y_true_intent, y_pred_intent):.4f}")
    print(f"Macro F1: {f1_score(y_true_intent, y_pred_intent, average='macro'):.4f}")
    print(f"Kappa: {cohen_kappa_score(y_true_intent, y_pred_intent):.4f}")

    print("\nSlot filling metrics:")
    print(f"Precision: {precision_score(y_true_slots, y_pred_slots):.4f}")
    print(f"Recall: {recall_score(y_true_slots, y_pred_slots):.4f}")
    print(f"F1 score: {seq_f1_score(y_true_slots, y_pred_slots):.4f}")

    print("\nDetailed classification report:")
    print(classification_report(y_true_slots, y_pred_slots))


Cell 8: Run Zero-Shot and Few-Shot Evaluation

In [13]:
with open("/content/drive/MyDrive/intent_project/data/ar-SA.jsonl", 'r', encoding='utf-8') as f:
    print(json.loads(next(f).strip()))


{'id': '0', 'locale': 'ar-SA', 'partition': 'test', 'scenario': 'alarm', 'intent': 'alarm_set', 'utt': 'صحيني خمسة الفجر هذا الأسبوع', 'annot_utt': 'صحيني [time : خمسة الفجر] [date : هذا الأسبوع]', 'worker_id': '10', 'slot_method': [{'slot': 'time', 'method': 'translation'}, {'slot': 'date', 'method': 'translation'}], 'judgments': [{'worker_id': '7', 'intent_score': 1, 'slots_score': 1, 'grammar_score': 4, 'spelling_score': 2, 'language_identification': 'target'}, {'worker_id': '16', 'intent_score': 1, 'slots_score': 1, 'grammar_score': 3, 'spelling_score': 2, 'language_identification': 'target'}, {'worker_id': '6', 'intent_score': 1, 'slots_score': 1, 'grammar_score': 4, 'spelling_score': 2, 'language_identification': 'target'}]}


In [19]:
# Extract utterances to run batch predictions
texts_zeroshot = [ex['utt'] for ex in test_data_zeroshot]

# Run batch predictions
print("Running zero-shot batch predictions...")
pred_intents_zeroshot, _ = predict_intents_batch(texts_zeroshot)
pred_slots_zeroshot = predict_slots_batch(texts_zeroshot)

# Save zero-shot results
save_results('/content/drive/MyDrive/intent_project/splits/zero_shot_intents.json', pred_intents_zeroshot)
save_results('/content/drive/MyDrive/intent_project/splits/zero_shot_slots.json', pred_slots_zeroshot)

# Evaluate zero-shot results
print("Evaluating zero-shot results...")
evaluate_with_preds(test_data_zeroshot, pred_intents_zeroshot, pred_slots_zeroshot)

# Few-shot fine-tuning and evaluation steps remain unchanged


Running zero-shot batch predictions...


KeyboardInterrupt: 